## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# NRMS Baseline

Encode title-token sequences through word embeddings, multi-head self-attention and additive attention pooling. Encode the user's sequence of clicked news vectors with the corresponding attention architecture. Score each candidate by

$$s(u,a)=\mathbf u^\top\mathbf r_a,\qquad P(a\mid C_u)=\frac{e^{s(u,a)}}{\sum_{j\in C_u}e^{s(u,j)}}.$$

## Inputs and training

Load prepared title indices, GloVe initialization and histories; padding remains zero. The declared architecture uses 16 heads over a 256-dimensional representation. Train against one positive and four within-impression negatives, with replacement for short negative lists as in the source. Select the checkpoint on validation nDCG@5 and evaluate held-out full slates.

## Diagnostics

Training curves, user-history strata, hourly results and attention plots describe the current measured run. The source evaluator uses first-clicked reciprocal rank and its original AUC/tie conventions. An empty history lacks personalized reading evidence. This baseline has no invented article-cold evaluation; compare it to gated/reference experiments only when evaluator and prepared inputs match.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import ndcg_at_k, history_groups


In [ ]:
# Hardware and Environment Verification
import os
import sys
import re
import math
import time
import random
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configure Matplotlib styling
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 65)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(
        f"Device Memory:    {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )

print(f"Active Device:    {device}")
print("=" * 65)


In [ ]:
# Project Root & Path Alignment (Works seamlessly from both root and models/)
DATA_DIR = str(PATHS["raw"])
MODELS_DIR = str(PATHS["outputs"] / "base_models" / "nrms")
CHECKPOINTS_DIR = os.path.join(MODELS_DIR, "checkpoints")
FIGURES_DIR = os.path.join(MODELS_DIR, "figures")
SUMMARY_PATH = os.path.join(FIGURES_DIR, "results_summary.json")
# NRMS Model and Experimental Hyperparameters
CONFIG = {
    # Architecture Dimensions
    "word_embed_dim": 300,  # GloVe 6B 300d embeddings
    "d_model": 256,  # News and User representation dimension (16 heads * 16 dim)
    "num_heads": 16,  # Number of attention heads in MHSA
    "head_dim": 16,  # Dimension per attention head
    "query_dim": 200,  # Additive Attention query vector dimension
    "dropout": 0.2,  # Dropout rate applied after embeddings and attention
    # Sequence Length Constraints
    "max_title_len": 30,  # Maximum title sequence length (padded / truncated)
    "max_hist_len": 50,  # Maximum user click history length (padded / truncated)
    # Training & Negative Sampling Setup
    "neg_sampling_ratio": 4,  # 1 positive : 4 sampled negatives per clicked candidate
    "batch_size": 128,  # Training batch size
    "learning_rate": 1e-4,  # Adam optimizer learning rate
    "epochs": 3,  # Training epochs
    "seed": 42,  # Random seed for reproducibility
    # File Paths (Dynamically resolved based on notebook location)
    "train_news_path": os.path.join(DATA_DIR, "train/news.tsv"),
    "dev_news_path": os.path.join(DATA_DIR, "dev/news.tsv"),
    "train_behaviors_path": os.path.join(DATA_DIR, "train/behaviors.tsv"),
    "dev_behaviors_path": os.path.join(DATA_DIR, "dev/behaviors.tsv"),
    "glove_path": os.path.join(DATA_DIR, "glove/glove.6B.300d.txt"),
    "checkpoint_dir": CHECKPOINTS_DIR,
    "figures_dir": FIGURES_DIR,
    "summary_path": SUMMARY_PATH,
}
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["figures_dir"], exist_ok=True)
print("Active Configuration:")

for k, v in CONFIG.items():
    print(f"  {k:<22}: {v}")


## 1. Prepared title and history inputs

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
from types import SimpleNamespace

payload = pd.read_pickle(PATHS["features"] / "nrms_features.pkl")
data = SimpleNamespace(**payload)
news_id2idx = data.news_id2idx
news_idx2id = {idx: nid for nid, idx in news_id2idx.items()}
news_title_lookup = torch.as_tensor(
    data.news_title_lookup, dtype=torch.long, device=device
)
pretrained_embeddings = torch.as_tensor(data.pretrained_embeddings, dtype=torch.float32)
all_news = data.all_news
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
val_records, test_records = data.val_records, data.test_records
max_title_len = data.max_title_len


def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower()) if isinstance(text, str) else []


In [ ]:
# 5.2 Construct Training and Evaluation Datasets
print("[Step 4] Constructing Datasets with Impression-Constrained Negative Sampling...")
max_hist_len = CONFIG["max_hist_len"]
# 1. Training Samples (1 Pos + 4 Negatives sampled per click within impression)
train_samples = []

for row in train_df.itertuples(index=False):
    raw_h = row.history.split() if isinstance(row.history, str) else []
    hist = [news_id2idx[x] for x in raw_h if x in news_id2idx][-max_hist_len:]
    if len(hist) < max_hist_len:
        hist = [0] * (max_hist_len - len(hist)) + hist
    pos, neg = [], []
    for item in row.impressions.split():
        nid, lbl = item.split("-")
        if nid in news_id2idx:
            if lbl == "1":
                pos.append(news_id2idx[nid])
            else:
                neg.append(news_id2idx[nid])
    if not pos or not neg:
        continue
    for p in pos:
        sampled_neg = (
            random.sample(neg, 4) if len(neg) >= 4 else random.choices(neg, k=4)
        )
        train_samples.append((hist, [p] + sampled_neg))

print(f"  • Constructed {len(train_samples):,} training instances (1 pos : 4 neg).")


class NewsTrainDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        h, c = self.samples[idx]
        return torch.tensor(h, dtype=torch.long), torch.tensor(c, dtype=torch.long)


train_loader = DataLoader(
    NewsTrainDataset(train_samples),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    drop_last=True,
)


## 2. PyTorch NRMS Architecture Implementation

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
class AttentionPooling(nn.Module):
    """
    Additive Attention Layer (Attention Pooling) as formulated in NRMS (Wu et al., 2019):
        a_i = q^T tanh(W h_i + b)
        alpha_i = exp(a_i) / sum(exp(a_j))
        output = sum(alpha_i * h_i)
    """

    def __init__(self, d_model=256, query_dim=200):
        super().__init__()
        self.proj = nn.Linear(d_model, query_dim)
        self.query = nn.Parameter(torch.empty(query_dim).uniform_(-0.1, 0.1))

    def forward(self, x, mask=None):
        energy = torch.tanh(self.proj(x))  # [B, L, query_dim]
        scores = torch.matmul(energy, self.query)  # [B, L]
        if mask is not None:
            all_pad = (~mask).all(dim=-1, keepdim=True)
            scores = scores.masked_fill(~mask, -1e9)
        alpha = F.softmax(scores, dim=-1)  # [B, L]
        if mask is not None:
            alpha = alpha.masked_fill(all_pad, 0.0)
        out = torch.sum(x * alpha.unsqueeze(-1), dim=1)  # [B, D]
        return out, alpha


class NewsEncoder(nn.Module):
    """
    News Encoder:
    Word Tokens -> GloVe Embedding (300d) -> Projection (256d) -> MHSA (16 heads) -> Additive Attention (200d query)
    """

    def __init__(
        self,
        pretrained_embed,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_embed, freeze=False, padding_idx=0
        )
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(pretrained_embed.shape[1], d_model)
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, tokens, mask=None, return_weights=False):
        if mask is None:
            mask = tokens > 0
        x = self.embedding(tokens)
        x = self.dropout(x)
        x = self.proj(x)
        if self.use_self_attn:
            key_padding_mask = ~mask
            all_pad = key_padding_mask.all(dim=-1)
            if all_pad.any():
                key_padding_mask = key_padding_mask.clone()
                key_padding_mask[all_pad] = False
            attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
            attn_out = self.dropout(attn_out)
        else:
            attn_out = x
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


In [ ]:
class UserEncoder(nn.Module):
    """
    User Encoder:
    Historical News Vectors -> News MHSA (16 heads) -> Additive Attention (200d query)
    Handles cold-start (zero click history) users gracefully.
    """

    def __init__(
        self, d_model=256, n_heads=16, query_dim=200, dropout=0.2, use_self_attn=True
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, news_vecs, mask=None, return_weights=False):
        if self.use_self_attn:
            if mask is not None:
                key_padding_mask = ~mask
                all_pad = key_padding_mask.all(dim=-1)
                if all_pad.any():
                    key_padding_mask = key_padding_mask.clone()
                    key_padding_mask[all_pad] = False
            else:
                key_padding_mask = None
            attn_out, _ = self.self_attn(
                news_vecs, news_vecs, news_vecs, key_padding_mask=key_padding_mask
            )
            attn_out = self.dropout(attn_out)
        else:
            attn_out = news_vecs
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


In [ ]:
class NRMS(nn.Module):
    """
    Neural News Recommendation with Multi-Head Self-Attention (NRMS)
    Features weight-sharing between candidate and history news encoders,
    and high-throughput unique-ID batch indexing.
    """

    def __init__(
        self,
        pretrained_embed,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
    ):
        super().__init__()
        self.register_buffer("news_title_lookup", news_title_lookup)
        self.news_encoder = NewsEncoder(
            pretrained_embed,
            d_model,
            n_heads,
            query_dim,
            dropout,
            use_self_attn=use_self_attn,
        )
        self.user_encoder = UserEncoder(
            d_model, n_heads, query_dim, dropout, use_self_attn=use_self_attn
        )

    def forward(self, hist_ids, cand_ids):
        B, P = hist_ids.shape
        C = cand_ids.shape[1]
        # High-throughput optimization: encode only unique news titles in the batch
        all_ids = torch.cat([hist_ids.view(-1), cand_ids.view(-1)])
        unique_ids, inv = torch.unique(all_ids, return_inverse=True)
        unique_tokens = self.news_title_lookup[unique_ids]
        unique_vecs = self.news_encoder(unique_tokens)
        all_vecs = unique_vecs[inv]
        hist_news_vecs = all_vecs[: B * P].view(B, P, -1)
        cand_news_vecs = all_vecs[B * P :].view(B, C, -1)
        hist_mask = hist_ids > 0
        user_vecs = self.user_encoder(hist_news_vecs, mask=hist_mask)
        # Dot product click prediction: s(u, c) = u^T e_c
        scores = torch.bmm(cand_news_vecs, user_vecs.unsqueeze(2)).squeeze(2)
        return scores


print("NRMS Architecture defined successfully.")


## 3. Fast Ranking Evaluation Engine & Metrics

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
def eval_impression(labels, scores):
    labels = np.asarray(labels, dtype=np.int32)
    scores = np.asarray(scores, dtype=np.float32)
    pos_mask = labels == 1
    n_pos = pos_mask.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        auc = 0.5
    else:
        order = np.argsort(scores)
        ranks = np.empty_like(order)
        ranks[order] = np.arange(1, len(scores) + 1)
        auc = (ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    desc_order = np.argsort(-scores)
    sorted_labels = labels[desc_order]
    pos_ranks = np.where(sorted_labels == 1)[0]
    mrr = 1.0 / (pos_ranks[0] + 1) if len(pos_ranks) > 0 else 0.0
    return (
        float(auc),
        float(mrr),
        ndcg_at_k(labels, sorted_labels, 5),
        ndcg_at_k(labels, sorted_labels, 10),
    )


In [ ]:
@torch.no_grad()
def evaluate_model(model, eval_records, news_title_lookup, device, batch_size=2048):
    model.eval()
    num_news = len(news_title_lookup)
    news_vecs = torch.zeros(num_news, 256, device=device)
    # 1. Pre-encode all unique news articles once
    for i in range(0, num_news, batch_size):
        chunk = news_title_lookup[i : min(i + batch_size, num_news)].to(device)
        vecs = model.news_encoder(chunk)
        news_vecs[i : min(i + batch_size, num_news)] = vecs
    news_vecs[0] = 0.0
    # 2. Batch encode all users
    num_eval = len(eval_records)
    hist_matrix = torch.zeros(num_eval, 50, dtype=torch.long, device=device)
    for idx, rec in enumerate(eval_records):
        hist_matrix[idx] = torch.tensor(rec["history"], dtype=torch.long, device=device)
    user_vecs = torch.zeros(num_eval, 256, device=device)
    user_batch_size = 1024
    for i in range(0, num_eval, user_batch_size):
        chunk_hist = hist_matrix[i : min(i + user_batch_size, num_eval)]
        chunk_news_vecs = news_vecs[chunk_hist]
        mask = chunk_hist > 0
        u_vecs = model.user_encoder(chunk_news_vecs, mask=mask)
        user_vecs[i : min(i + user_batch_size, num_eval)] = u_vecs
    # 3. Compute dot products and ranking metrics per impression
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []
    detailed_results = []
    for i, rec in enumerate(eval_records):
        cands = rec["candidates"]
        labels = rec["labels"]
        u_vec = user_vecs[i]
        c_vecs = news_vecs[cands]
        scores = torch.mv(c_vecs, u_vec).cpu().numpy()
        auc, mrr, n5, n10 = eval_impression(labels, scores)
        aucs.append(auc)
        mrrs.append(mrr)
        ndcg5s.append(n5)
        ndcg10s.append(n10)
        detailed_results.append(
            {
                "impression_id": rec["impression_id"],
                "timestamp": rec["timestamp"],
                "history_len": rec["history_len"],
                "hour": (
                    rec["timestamp"].hour if hasattr(rec["timestamp"], "hour") else 0
                ),
                "auc": auc,
                "mrr": mrr,
                "ndcg5": n5,
                "ndcg10": n10,
            }
        )
    return {
        "auc": float(np.mean(aucs)),
        "mrr": float(np.mean(mrrs)),
        "ndcg5": float(np.mean(ndcg5s)),
        "ndcg10": float(np.mean(ndcg10s)),
        "detailed": detailed_results,
    }


print("Evaluation engine compiled successfully.")


## 4. NRMS Model Training & Validation Execution

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
# 8.1 Model Initialization and Training Execution
model = NRMS(
    pretrained_embeddings,
    news_title_lookup,
    d_model=CONFIG["d_model"],
    n_heads=CONFIG["num_heads"],
    query_dim=CONFIG["query_dim"],
    dropout=CONFIG["dropout"],
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
best_ndcg5 = -1.0
best_model_path = os.path.join(CONFIG["checkpoint_dir"], "nrms_best_seed42.pt")
history = {
    "train_loss": [],
    "val_auc": [],
    "val_mrr": [],
    "val_ndcg5": [],
    "val_ndcg10": [],
}
# Load pre-trained weights if available, or train from scratch

if os.path.exists(best_model_path):
    print(f"Loading cached best model weights from {best_model_path}...")
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    if os.path.exists(CONFIG["summary_path"]):
        with open(CONFIG["summary_path"]) as f:
            res_json = json.load(f)
            history = res_json.get("history", history)
            print("Loaded training history logs:")
            for ep in range(len(history["train_loss"])):
                print(
                    f"  Epoch {ep+1}/3 | Train Loss: {history['train_loss'][ep]:.4f} | "
                    f"Val AUC: {history['val_auc'][ep]:.4f} | MRR: {history['val_mrr'][ep]:.4f} | "
                    f"NDCG@5: {history['val_ndcg5'][ep]:.4f} | NDCG@10: {history['val_ndcg10'][ep]:.4f}"
                )
else:
    print("Training NRMS from scratch...")
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        total_loss = 0.0
        t_epoch = time.time()
        for step, (hist_b, cand_b) in enumerate(train_loader):
            hist_b = hist_b.to(device)
            cand_b = cand_b.to(device)
            optimizer.zero_grad()
            scores = model(hist_b, cand_b)
            targets = torch.zeros(scores.shape[0], dtype=torch.long, device=device)
            loss = F.cross_entropy(scores, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        val_metrics = evaluate_model(model, val_records, news_title_lookup, device)
        history["train_loss"].append(avg_loss)
        history["val_auc"].append(val_metrics["auc"])
        history["val_mrr"].append(val_metrics["mrr"])
        history["val_ndcg5"].append(val_metrics["ndcg5"])
        history["val_ndcg10"].append(val_metrics["ndcg10"])
        print(
            f"Epoch {epoch}/{CONFIG['epochs']} | Loss: {avg_loss:.4f} | "
            f"Val AUC: {val_metrics['auc']:.4f} | MRR: {val_metrics['mrr']:.4f} | "
            f"NDCG@5: {val_metrics['ndcg5']:.4f} | NDCG@10: {val_metrics['ndcg10']:.4f}"
        )
        if val_metrics["ndcg5"] > best_ndcg5:
            best_ndcg5 = val_metrics["ndcg5"]
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Saved new best checkpoint to {best_model_path}")
    model.load_state_dict(torch.load(best_model_path, map_location=device))


In [ ]:
# 8.2 Visualize Training Convergence and Validation Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)
# Cross-entropy loss curve
axes[0].plot(
    epochs_range,
    history["train_loss"],
    marker="o",
    color="#2b5c8f",
    lw=2.5,
    label="Train Cross-Entropy Loss",
)
axes[0].set_title("NRMS Training Loss vs. Epoch", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Cross-Entropy Loss", fontsize=12)
axes[0].set_xticks(list(epochs_range))
axes[0].legend(frameon=True)
axes[0].grid(True, linestyle="--", alpha=0.6)
# Validation metrics
axes[1].plot(
    epochs_range, history["val_auc"], marker="s", color="#2ca02c", lw=2, label="Val AUC"
)
axes[1].plot(
    epochs_range,
    history["val_ndcg5"],
    marker="^",
    color="#d62728",
    lw=2,
    label="Val NDCG@5",
)
axes[1].plot(
    epochs_range, history["val_mrr"], marker="d", color="#9467bd", lw=2, label="Val MRR"
)
axes[1].plot(
    epochs_range,
    history["val_ndcg10"],
    marker="v",
    color="#ff7f0e",
    lw=2,
    label="Val NDCG@10",
)
axes[1].set_title("Validation Ranking Metrics (Day 6)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Metric Score", fontsize=12)
axes[1].set_xticks(list(epochs_range))
axes[1].legend(frameon=True)
axes[1].grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "training_curves.png"), dpi=300)
plt.show()


## 5. Experiment 1: Original NRMS Benchmark Evaluation (Test Split)

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
# 9.1 Evaluate on Day 7 Test Split (73,152 Impressions)
print("Evaluating best checkpoint on Test Split (Day 7)...")
t_test = time.time()
test_results = evaluate_model(model, test_records, news_title_lookup, device)
eval_time = time.time() - t_test
benchmark_df = pd.DataFrame(
    [
        {
            "Model Architecture": "Original NRMS (Wu et al., 2019)",
            "Test Impressions": len(test_records),
            "AUC": f"{test_results['auc']:.4f}",
            "MRR": f"{test_results['mrr']:.4f}",
            "NDCG@5": f"{test_results['ndcg5']:.4f}",
            "NDCG@10": f"{test_results['ndcg10']:.4f}",
            "Eval Latency": f"{eval_time:.2f}s",
        }
    ]
)
print("\n" + "=" * 75)
print("TEST BENCHMARK RESULTS (MIND-SMALL DAY 7)")
print("=" * 75)
print(benchmark_df.to_string(index=False))
print("=" * 75)
# Synchronize live evaluation results into summary JSON
os.makedirs(CONFIG["figures_dir"], exist_ok=True)
summary_path = CONFIG["summary_path"]
summary_dict = {}

if os.path.exists(summary_path):
    try:
        with open(summary_path) as f:
            summary_dict = json.load(f)
    except Exception:
        summary_dict = {}

summary_dict["single_seed"] = {
    "auc": float(test_results["auc"]),
    "mrr": float(test_results["mrr"]),
    "ndcg5": float(test_results["ndcg5"]),
    "ndcg10": float(test_results["ndcg10"]),
}

if "history" in locals():
    summary_dict["history"] = history

with open(summary_path, "w") as f:
    json.dump(summary_dict, f, indent=2)

print(f"--> Successfully synchronized Seed 42 benchmark into {summary_path}")


## 6. Experiment 3: User Cold-Start Stratification Analysis

Evaluate cold users with 0 historical articles, sparse users with 1–4, and warm users with at least 5.


In [ ]:
# 11.1 Stratified Cold-Start Evaluation Table
detailed_df = pd.DataFrame(test_results["detailed"])
bins = [-1, 0, 4, float("inf")]
labels = ["Cold Start (0)", "Sparse (1-4)", "Warm (>=5)"]
detailed_df["history_bin"] = pd.cut(
    detailed_df["history_len"], bins=bins, labels=labels
)
cold_start_rows = []

for b in labels:
    sub = detailed_df[detailed_df["history_bin"] == b]
    cold_start_rows.append(
        {
            "History Segment": b,
            "Impressions": len(sub),
            "Pct (%)": f"{len(sub)/len(detailed_df)*100:.1f}%",
            "AUC": f"{sub['auc'].mean():.4f}",
            "MRR": f"{sub['mrr'].mean():.4f}",
            "NDCG@5": f"{sub['ndcg5'].mean():.4f}",
            "NDCG@10": f"{sub['ndcg10'].mean():.4f}",
        }
    )

cold_start_df = pd.DataFrame(cold_start_rows)
print("\n" + "=" * 80)
print("USER COLD-START RANKING PERFORMANCE BY HISTORY LENGTH")
print("=" * 80)
print(cold_start_df.to_string(index=False))
print("=" * 80)


In [ ]:
# 11.2 Cold-Start Performance Visualization
fig, ax = plt.subplots(figsize=(10, 5.5))
cs_plot_df = pd.DataFrame(
    {
        "History Length": labels,
        "AUC": [
            detailed_df[detailed_df["history_bin"] == b]["auc"].mean() for b in labels
        ],
        "MRR": [
            detailed_df[detailed_df["history_bin"] == b]["mrr"].mean() for b in labels
        ],
        "NDCG@5": [
            detailed_df[detailed_df["history_bin"] == b]["ndcg5"].mean() for b in labels
        ],
        "NDCG@10": [
            detailed_df[detailed_df["history_bin"] == b]["ndcg10"].mean()
            for b in labels
        ],
    }
)
x = np.arange(len(labels))
width = 0.2
ax.bar(x - 1.5 * width, cs_plot_df["AUC"], width, label="AUC", color="#1f77b4")
ax.bar(x - 0.5 * width, cs_plot_df["MRR"], width, label="MRR", color="#ff7f0e")
ax.bar(x + 0.5 * width, cs_plot_df["NDCG@5"], width, label="NDCG@5", color="#2ca02c")
ax.bar(x + 1.5 * width, cs_plot_df["NDCG@10"], width, label="NDCG@10", color="#d62728")
ax.set_title(
    "NRMS Ranking Performance across User History Lengths",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("User Interaction History Length", fontsize=12)
ax.set_ylabel("Metric Score", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0.15, 0.75)
ax.legend(frameon=True, loc="upper left")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "cold_start_analysis.png"), dpi=300)
plt.show()


## 7. Experiment 4: Diurnal Temporal Stability Analysis

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
# 12.1 Hourly Diurnal Performance Curve
hourly_summary = (
    detailed_df.groupby("hour")[["auc", "mrr", "ndcg5", "ndcg10"]].mean().reset_index()
)
fig, ax1 = plt.subplots(figsize=(12, 5.5))
color1 = "#1f77b4"
ax1.set_xlabel("Hour of Day (UTC)", fontsize=12)
ax1.set_ylabel("NDCG@5", color=color1, fontsize=12)
line1 = ax1.plot(
    hourly_summary["hour"],
    hourly_summary["ndcg5"],
    color=color1,
    marker="o",
    lw=2.5,
    label="NDCG@5",
)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_xticks(range(0, 24))
ax1.grid(True, linestyle="--", alpha=0.6)
ax2 = ax1.twinx()
color2 = "#d62728"
ax2.set_ylabel("AUC", color=color2, fontsize=12)
line2 = ax2.plot(
    hourly_summary["hour"],
    hourly_summary["auc"],
    color=color2,
    marker="s",
    lw=2.5,
    linestyle="--",
    label="AUC",
)
ax2.tick_params(axis="y", labelcolor=color2)
lines = line1 + line2
labels_leg = [l.get_label() for l in lines]
ax1.legend(lines, labels_leg, loc="lower right", frameon=True)
plt.title(
    "Diurnal Ranking Performance Stability across Day 7 Test Hours",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(
    os.path.join(CONFIG["figures_dir"], "temporal_hourly_analysis.png"), dpi=300
)
plt.show()
print("Hourly Ranking Stability Analysis:")
print(
    f"  • NDCG@5 Range: [{hourly_summary['ndcg5'].min():.4f}, {hourly_summary['ndcg5'].max():.4f}] (Std: {hourly_summary['ndcg5'].std():.4f})"
)
print(
    f"  • AUC Range:    [{hourly_summary['auc'].min():.4f}, {hourly_summary['auc'].max():.4f}] (Std: {hourly_summary['auc'].std():.4f})"
)


## 8. Experiment 5: Model Interpretability & Attention Visualization

Configure and evaluate the declared NRMS experiment using prepared title and history inputs.


In [ ]:
# 13.1 Word-Level Additive Attention Visualization
sample_news_ids = list(all_news["news_id"].iloc[10:14])
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for idx, snid in enumerate(sample_news_ids):
    n_idx = news_id2idx[snid]
    toks = news_title_lookup[n_idx : n_idx + 1]
    with torch.no_grad():
        _, alpha = model.news_encoder(toks, return_weights=True)
    alpha_np = alpha[0].detach().cpu().numpy()
    raw_title = all_news[all_news["news_id"] == snid]["title"].values[0]
    title_words = tokenize(raw_title)[:max_title_len]
    alpha_words = alpha_np[: len(title_words)]
    if alpha_words.sum() > 0:
        alpha_words = alpha_words / alpha_words.sum()
    y_pos = np.arange(len(title_words))
    axes[idx].barh(y_pos, alpha_words, color="#2b5c8f", alpha=0.85)
    axes[idx].set_yticks(y_pos)
    axes[idx].set_yticklabels(title_words, fontsize=10)
    axes[idx].invert_yaxis()
    axes[idx].set_title(f'"{raw_title[:38]}..."', fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("Additive Attention Weight", fontsize=10)
    axes[idx].grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    "Word-Level Additive Attention Weights on Candidate News Titles",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["figures_dir"], "attention_visualization.png"), dpi=300)
plt.show()
